In [1]:
import pandas as pd
import json

# Define the dataset path
full_text_dataset_path = "/Users/cfh00907720/Documents/Citation Recommendation/RAG/global_train_eval_context_list_for_rag.json"

# Load the JSON file
with open(full_text_dataset_path, "r", encoding="utf-8") as file:
    data = json.load(file)

# Convert JSON data to a Pandas DataFrame
df = pd.DataFrame(data)

In [2]:
df

,acl_id,author,title_on_paper,year_on_paper,abstract,full_text,citation_item,title_on_citation_list,year_on_citation_list
0,P98-1013,"Baker, Collin F. and Fillmore, Charles J. and ...",The {B}erkeley {F}rame{N}et Project,1998,FrameNet is a three-year NSF-supported project...,FrameNet is a three-year NSF-supported project...,"Baker et al., 1998",The Berkeley FrameNet Project,1998
1,P05-1018,"Barzilay, Regina and Lapata, Mirella",Modeling Local Coherence: An Entity-Based Appr...,2005,This paper considers the problem of automatic ...,This paper considers the problem of automatic ...,"Barzilay and Lapata, 2008",Modeling Local Coherence: An Entity-based Appr...,2005
2,P00-1037,"Brill, Eric and Moore, Robert C.",An Improved Error Model for Noisy Channel Spel...,2000,The noisy channel model has been applied to a ...,The noisy channel model has been applied to a ...,"Brill and Moore, 2000",An Improved Error Model for Noisy Channel Spel...,2000
3,P08-1093,"Mei, Qiaozhu and Zhai, ChengXiang",Generating Impact-Based Summaries for Scientif...,2008,"In this paper, we present a study of a novel s...","In this paper, we present a study of a novel s...","Mei and Zhai, 2008",Generating Impact-Based Summaries for Scientif...,2008
4,J07-2003,"Chiang, David",Hierarchical Phrase-Based Translation,2007,The alignment template translation model (Och ...,We present a statistical machine translation m...,"Chiang, 2007",Hierarchical Phrase-Based Translation,2007
...,...,...,...,...,...,...,...,...,...
5241,P09-2074,"Daumé III, Hal",Markov Random Topic Fields,2009,Most approaches to topic modeling assume an in...,Proceedings of the ACL-IJCNLP 2009 Conference ...,"Daume, 2009",Markov Random Topic Fields,2009
5242,J96-1006,"Covington, Michael A.",Natural Language Processing for Prolog Program...,1996,Ken Barker is a doctoral student in computatio...,Book Reviews \nNatural Language Processing for...,"Covington, 1994",Natural Language Processing for Prolog Program...,1994
5243,P07-2042,"Sang, Erik",Extracting Hypernym Pairs from the Web,2007,We apply pattern-based methods for collecting ...,Proceedings of the ACL 2007 Demo and Poster Se...,"Sang, 2007",Extracting Hypernym Pairs from the Web,2007
5244,N06-2004,"Klebanov, Beata Beigman",Measuring Semantic Relatedness Using People an...,2006,"In this paper, we (1) propose a new dataset fo...",Proceedings of the Human Language Technology C...,"Klebanov, 2006",Measuring Semantic Relatedness Using People an...,2006


In [3]:
import re
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

# Function to check if full_text contains intelligible content
def is_unintelligible(text):
    if not isinstance(text, str) or not text.strip():
        return True  # Empty or non-string text is unintelligible
    
    # Remove common punctuation and check if any words remain
    cleaned_text = re.sub(r'[^a-zA-Z0-9À-ÖØ-öø-ÿ]', ' ', text).strip()

    if len(cleaned_text) < 5:  # Arbitrary threshold to remove very short meaningless texts
        return True

    # Attempt language detection
    try:
        lang = detect(cleaned_text)
        if lang in ['en', 'fr', 'es', 'de', 'zh-cn', 'zh-tw']:  # Adjust languages as needed
            return False  # Considered intelligible
    except LangDetectException:
        return True  # Could not detect a valid language

    return True  # Default to unintelligible if it doesn't match criteria

# Apply the function to flag unintelligible entries
df['unintelligible'] = df['full_text'].apply(is_unintelligible)

# Filter out unintelligible entries
unintelligible_texts = df[df['unintelligible']]

In [4]:
# Define the dataset path
global_context_rag_path = "/Users/cfh00907720/Documents/Citation Recommendation/RAG/global_eval_context_for_rag.json"

# Load the global_eval_context dataset
with open(global_context_rag_path, "r", encoding="utf-8") as file:
    global_context_data = json.load(file)

# Convert to DataFrame
global_context_df = pd.DataFrame(global_context_data)

# Ensure "target_title" exists
if "target_title" not in global_context_df.columns:
    raise KeyError("The dataset does not contain 'target_title' column.")

# Count occurrences of each title_on_paper in target_title
title_counts = global_context_df["target_title"].value_counts().to_dict()

# Ensure unintelligible_texts is a copy before modifications
unintelligible_texts = unintelligible_texts.copy()

# Use .loc to modify specific columns safely
unintelligible_texts.loc[:, "count_in_eval_context"] = unintelligible_texts["title_on_paper"].map(title_counts).fillna(0).astype(int)

# Calculate the percentage using .loc
total_titles = global_context_df.shape[0]  # Total number of entries in global_eval_context
unintelligible_texts.loc[:, "percentage_of_usage"] = (unintelligible_texts["count_in_eval_context"] / total_titles) * 100

In [5]:
unintelligible_texts

,acl_id,author,title_on_paper,year_on_paper,abstract,full_text,citation_item,title_on_citation_list,year_on_citation_list,unintelligible,count_in_eval_context,percentage_of_usage
19,N01-1008,"Harabagiu, Sanda M. and Bunescu, Razvan C. and...",Text and Knowledge Mining for Coreference Reso...,2001,Traditionally coreference is resolved by satis...,P TS VU XW YP abP ac dP fe hg pi q sr ut P av ...,"Harabagiu et al., 2001",Text and Knowledge Mining for Coreference Reso...,2001,True,3,0.023827
21,W04-3212,"Xue, Nianwen and Palmer, Martha",Calibrating Features for Semantic Role Labeling,2004,This paper takes a critical look at the featur...,"dc fe ""g ih qp sr ut ug iv xw y p se ""e ""t ¦w ...","Xue and Palmer, 2004",Calibrating Features for Semantic Role Labeling,2004,True,25,0.198555
40,W01-0701,"Florian, Radu and Ngai, Grace",Multidimensional transformation-based learning,2001,This paper presents a novel method that allows...,B DC 0E GF IH 3P RQ TS VU WC 0X `Y ba dc @e gf...,"Florian and Ngai, 2001",Multidimensional Transformation-Based Learning,2001,True,0,0.000000
85,P00-1056,"Och, Franz Josef and Ney, Hermann",Improved Statistical Alignment Models,2000,"In this paper, we present and compare various ...",¥ §¦ T ©E GF ¨a hä « 2¬ $ ¢® C°%± °&² g³ µ @² ...,"Och and Ney, 2000",Improved Statistical Alignment Models,2000,True,57,0.452704
187,P02-1031,"Gildea, Daniel and Palmer, Martha",The Necessity of Parsing for Predicate Argumen...,2002,Broad-coverage corpora annotated with semantic...,×ØÖ Ø ÖÓ ¹ ÓÚ Ö ÓÖÔÓÖ ÒÒÓØ Ø Û Ø × Ñ ÒØ ÖÓÐ ¸Ó...,"Gildea and Palmer, 2002",The Necessity of Parsing for Predicate Argumen...,2002,True,10,0.079422
...,...,...,...,...,...,...,...,...,...,...,...,...
5237,W03-2605,"Poesio, Massimo",Associative Descriptions and Salience: A Preli...,2003,We discuss the problems involved in identifyin...,\n\n\n\n\n\n\n\n,"Poesio, 2003",Associative Descriptions and Salience: A Preli...,2003,True,0,0.000000
5238,W01-0724,"Sang, Erik",Memory-based clause identification,2001,We apply a memory-based learner to the CoNLL-2..., \t \n\r    !#...,"Sang, 2001",Memory-Based Clause Identification,2001,True,0,0.000000
5239,W03-2409,"Sampson, Geoffrey and Babarczy, Anna",Limits to annotation precision,2003,This paper seeks to draw attention to a large ...,\n\n\n\n\n\n\n\n,"Sampson and Babarczy, 2003",Limits to annotation precision,2003,True,0,0.000000
5240,W03-2312,"Reiter, Ehud and Sripada, Somayajulu and Willi...",Acquiring and Using Limited User Models in {NLG},2003,It is a truism of NLG that good knowledge of t...,\n\n\n\n\n\n\n\n,"Reiter et al., 2003",Acquiring and Using Limited User Models in NLG,2003,True,0,0.000000


In [6]:
import json
import pandas as pd

# Define the dataset path
global_context_rag_path = "/Users/cfh00907720/Documents/Citation Recommendation/RAG/global_eval_context_for_rag.json"

# Load the global_eval_context dataset
with open(global_context_rag_path, "r", encoding="utf-8") as file:
    global_context_data = json.load(file)

# Convert to DataFrame
global_context_df = pd.DataFrame(global_context_data)

# Ensure "target_title" exists
if "target_title" not in global_context_df.columns:
    raise KeyError("The dataset does not contain 'target_title' column.")

# Count occurrences of each title_on_paper in target_title
title_counts = global_context_df["target_title"].value_counts()

# Sum all occurrences from the unintelligible_texts titles
total_unintelligible_occurrences = title_counts.reindex(unintelligible_texts["title_on_paper"]).fillna(0).sum()

# Get the total number of entries in global_eval_context
total_entries_in_eval = len(global_context_df)

# Compute overall percentage
percentage_of_unintelligible_usage = (total_unintelligible_occurrences / total_entries_in_eval) * 100

# Print result
print(f"Total occurrences of unintelligible titles in eval dataset: {total_unintelligible_occurrences}")
print(f"Percentage of unintelligible title usage in eval dataset: {percentage_of_unintelligible_usage:.2f}%")

Total occurrences of unintelligible titles in eval dataset: 252.0
Percentage of unintelligible title usage in eval dataset: 2.00%


In [7]:
# Assuming unintelligible_texts DataFrame already exists

# Store ACL IDs for dropping
acl_ids_to_drop = unintelligible_texts["acl_id"].tolist()

# Store citation_item to drop in eval dataset where count_in_eval_context > 0
citation_items_to_drop_eval = unintelligible_texts.loc[
    unintelligible_texts["count_in_eval_context"] > 0, "citation_item"
].tolist()

# Store citation_item to drop in train dataset where count_in_eval_context == 0
citation_items_to_drop_train = unintelligible_texts.loc[
    unintelligible_texts["count_in_eval_context"] == 0, "citation_item"
].tolist()

# Print results
print("ACL IDs to drop in full context list for RAG:", acl_ids_to_drop)
print("Citation items to drop in eval:", citation_items_to_drop_eval)
print("Citation items to drop in train:", citation_items_to_drop_train)

ACL IDs to drop in full context list for RAG: ['N01-1008', 'W04-3212', 'W01-0701', 'P00-1056', 'P02-1031', 'W02-1007', 'P00-1023', 'W01-0514', 'W03-1011', 'N01-1008', 'N01-1006', 'W11-0110', 'W03-1709', 'W01-1205', 'N01-1013', 'W02-0401', 'W01-0721', 'P04-3013', 'W03-1801', 'P00-1064', 'N01-1002', 'P02-1020', 'W01-0726', 'W02-1109', 'W02-2003', 'W02-2035', 'W04-0704', 'W04-2307', 'W02-0204', 'P00-1064', 'P01-1063', 'W02-2004', 'P00-1030', 'P00-1059', 'W03-2808', 'P00-1069', 'W01-0725', 'W01-1204', 'P03-1056', 'W01-1616', 'P00-1024', 'P00-1040', 'W02-2020', 'W02-0505', 'W01-1616', 'W02-2027', 'P00-1030', 'W04-2604', 'P00-1056', 'P02-1031', 'W03-1011', 'W02-2034', 'W04-3212', 'W01-0520', 'W01-1621', 'W04-2216', 'W04-1109', 'W02-2035', 'W01-0726', 'P04-1003', 'W03-2707', 'W03-2401', 'W07-2315', 'P04-1003', 'W03-2502', 'W03-2506', 'W02-2025', 'W03-2606', 'W03-2607', 'W03-2316', 'W03-2403', 'W03-2317', 'W03-2314', 'W04-0909', 'W00-1606', 'W03-2605', 'W01-0724', 'W03-2409', 'W03-2312', 'W03-

### Dropping the Not Needed Citations

In [13]:
# Define the JSON file path
full_path = "/Users/cfh00907720/Documents/Citation Recommendation/RAG/[March 10, 2025] Final Version of Datasets/Full with Errors/global_train_eval_context_list_for_rag.json"

# Load the JSON file
with open(full_path, "r", encoding="utf-8") as file:
    data = json.load(file)

print(len(data))

# Convert to DataFrame
df = pd.DataFrame(data)

# Assuming acl_ids_to_drop is already defined as a list
# Example: acl_ids_to_drop = ["P95-1028", "P04-1082", "W99-0508"]

# Filter out rows where acl_id is in acl_ids_to_drop
df_filtered = df[~df["acl_id"].isin(acl_ids_to_drop)]

print(len(df_filtered))

# New file location
target_path = "/Users/cfh00907720/Documents/Citation Recommendation/RAG/[March 10, 2025] Final Version of Datasets/Cleaned No Errors/cleaned_global_train_eval_context_list.json"  

# Save the cleaned DataFrame back to JSON
df_filtered.to_json(target_path, orient="records", force_ascii=False, indent=4)

print(f"Entries with ACL IDs in acl_ids_to_drop have been removed. Updated JSON saved at {full_path}")

5246
5164
Entries with ACL IDs in acl_ids_to_drop have been removed. Updated JSON saved at /Users/cfh00907720/Documents/Citation Recommendation/RAG/[March 10, 2025] Final Version of Datasets/Full with Errors/global_train_eval_context_list_for_rag.json


In [14]:
# Define the JSON file path
full_path = "/Users/cfh00907720/Documents/Citation Recommendation/RAG/[March 10, 2025] Final Version of Datasets/Full with Errors/global_eval_context_for_rag.json"

# Load the JSON file
with open(full_path, "r", encoding="utf-8") as file:
    data = json.load(file)

print(len(data))

# Convert to DataFrame
df = pd.DataFrame(data)

# Filter out rows where masked_token_target is in citation_items_to_drop_eval
df_filtered = df[~df["masked_token_target"].isin(citation_items_to_drop_eval)]

print(len(df_filtered))

# New file location
target_path_json = "/Users/cfh00907720/Documents/Citation Recommendation/RAG/[March 10, 2025] Final Version of Datasets/Cleaned No Errors/cleaned_global_eval_context_for_rag.json"
target_path_csv = "/Users/cfh00907720/Documents/Citation Recommendation/RAG/[March 10, 2025] Final Version of Datasets/Cleaned No Errors/final_cleaned_acl_global_context_dataset_eval.csv"   

# Save the cleaned DataFrame back to JSON and CSV
df_filtered.to_json(target_path_json, orient="records", force_ascii=False, indent=4)
df_filtered.to_csv(target_path_csv, index=False, encoding="utf-8")

print("Saved the Files.")

12591
12450
Saved the Files.


In [15]:
# Define the JSON file path
full_path = "/Users/cfh00907720/Documents/Citation Recommendation/RAG/[March 10, 2025] Final Version of Datasets/Full with Errors/global_train_context_for_rag.json"

# Load the JSON file
with open(full_path, "r", encoding="utf-8") as file:
    data = json.load(file)

print(len(data))

# Convert to DataFrame
df = pd.DataFrame(data)

# Filter out rows where masked_token_target is in citation_items_to_drop_train and citation_items_to_drop_eval (since we don't want to train on items where eval shows up)
df_filtered = df[~df["masked_token_target"].isin(citation_items_to_drop_train)]
df_filtered = df_filtered[~df_filtered["masked_token_target"].isin(citation_items_to_drop_eval)]

print(len(df_filtered))

# New file location
target_path_json = "/Users/cfh00907720/Documents/Citation Recommendation/RAG/[March 10, 2025] Final Version of Datasets/Cleaned No Errors/cleaned_global_train_context_for_rag.json"
target_path_csv = "/Users/cfh00907720/Documents/Citation Recommendation/RAG/[March 10, 2025] Final Version of Datasets/Cleaned No Errors/final_cleaned_acl_global_context_dataset_train.csv"   

# Save the cleaned DataFrame back to JSON and CSV
df_filtered.to_json(target_path_json, orient="records", force_ascii=False, indent=4)
df_filtered.to_csv(target_path_csv, index=False, encoding="utf-8")

print("Saved the Files.")

49584
49367
Saved the Files.
